# gridMET Feature Builder

This notebook builds the final gridMET feature table for the wildfire severity project.

PyGridMET `coords` CSV outputs do **not** include a date column. Each raw file is already the 30-day pre-fire window requested. Therefore, the aggregation uses row position:
- last 7 rows = 7-day pre-fire window
- last 14 rows = 14-day pre-fire window
- all rows = 30-day pre-fire window


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

In [ ]:

def find_project_root() -> Path:
    """Find the repository root whether the notebook is run from root or notebooks/."""
    cwd = Path.cwd().resolve()

    if (cwd / "data").exists() and (cwd / "notebooks").exists():
        return cwd

    if cwd.name.lower() == "notebooks" and (cwd.parent / "data").exists():
        return cwd.parent

    for parent in [cwd] + list(cwd.parents):
        if (parent / "data").exists() and (parent / "notebooks").exists():
            return parent

    return cwd.parent if cwd.name.lower() == "notebooks" else cwd

PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
GRIDMET_OUTPUT_DIR = RAW_DIR / "gridmet_output"

for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw gridMET output:", GRIDMET_OUTPUT_DIR)
print("Processed output:", PROCESSED_DIR)


Project root: C:\Users\chaud\Desktop\repositories\wildfire-severity-v2
Raw gridMET output: C:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\gridmet_output
Processed output: C:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed


## 1. Load required input files

The two important metadata files are:
- `gridmet_requests.csv`: the original coordinate/date requests sent to PyGridMET
- `gridmet_fire_lookup.csv`: maps each `gridmet_id` back to the CAL FIRE incident


In [2]:
REQUESTS_PATH = INTERIM_DIR / "gridmet_requests.csv"
LOOKUP_PATH = INTERIM_DIR / "gridmet_fire_lookup.csv"

if not REQUESTS_PATH.exists():
    raise FileNotFoundError(f"Missing required file: {REQUESTS_PATH}")

if not LOOKUP_PATH.exists():
    raise FileNotFoundError(f"Missing required file: {LOOKUP_PATH}")

requests = pd.read_csv(REQUESTS_PATH)
lookup = pd.read_csv(LOOKUP_PATH)

required_request_cols = {"id", "lon", "lat", "start", "end"}
missing_request_cols = required_request_cols - set(requests.columns)

if missing_request_cols:
    raise ValueError(f"gridmet_requests.csv is missing required columns: {missing_request_cols}")

if "gridmet_id" not in lookup.columns:
    raise ValueError("gridmet_fire_lookup.csv must contain a 'gridmet_id' column.")

requests["id"] = requests["id"].astype(str)
lookup["gridmet_id"] = lookup["gridmet_id"].astype(str)

print("Requests:", requests.shape)
print("Lookup:", lookup.shape)

display(requests.head())
display(lookup.head())


Requests: (2404, 5)
Lookup: (2404, 7)


,id,lon,lat,start,end
0,fire_00000,-122.431720,38.409580,2016-09-10,2016-10-09
1,fire_00001,-121.080360,37.217100,2016-03-25,2016-04-23
2,fire_00002,-120.528836,37.927613,2016-04-30,2016-05-29
3,fire_00003,-121.200590,36.381230,2016-04-22,2016-05-21
4,fire_00004,-118.354000,34.276000,2016-04-23,2016-05-22


,gridmet_id,UniqueId,Name,StartedDateOnly,Latitude,Longitude,AcresBurned
0,fire_00000,3135c367-1274-4cc4-9152-661f6fd7707e,Creek Fire,2016-10-10,38.409580,-122.431720,65.0
1,fire_00001,7106ba43-100b-41f9-acbf-9fb9e9ab9ef7,Taglio Fire,2016-04-24,37.217100,-121.080360,30.0
2,fire_00002,ffec404e-dea9-4175-8c50-8f2b0ca3efdd,Tulloch Fire,2016-05-30,37.927613,-120.528836,85.0
3,fire_00003,55fa95e6-f09e-42c4-889b-a4b5b63dccf2,Metz Fire,2016-05-22,36.381230,-121.200590,3876.0
4,fire_00004,ace1f464-7a70-4dcc-b8c5-d64fb1021d0e,Wheatland Fire,2016-05-23,34.276000,-118.354000,156.0


## 2. Download coverage check

This compares expected request IDs against the raw CSV files already downloaded in `data/raw/gridmet_output/`.

It also writes small audit files:
- `data/interim/gridmet_missing_outputs.csv`
- `data/interim/gridmet_failed_ids_clean.csv`


In [3]:
downloaded_ids = {p.stem for p in GRIDMET_OUTPUT_DIR.glob("*.csv")}

requests_status = requests.copy()
requests_status["downloaded"] = requests_status["id"].isin(downloaded_ids)

missing_outputs = requests_status[~requests_status["downloaded"]].copy()
missing_outputs.to_csv(INTERIM_DIR / "gridmet_missing_outputs.csv", index=False)

failed_txt_path = INTERIM_DIR / "gridmet_failed_single_files.txt"
failed_ids = []

if failed_txt_path.exists():
    for line in failed_txt_path.read_text(encoding="utf-8").splitlines():
        match = re.search(r"(fire_\d+)", line)
        if match:
            failed_ids.append(match.group(1))

failed_ids = sorted(set(failed_ids))
failed_ids_clean = pd.DataFrame({"gridmet_id": failed_ids})
failed_ids_clean.to_csv(INTERIM_DIR / "gridmet_failed_ids_clean.csv", index=False)

print("Expected requests:", len(requests))
print("Downloaded raw CSVs:", len(downloaded_ids))
print("Missing raw CSVs:", len(missing_outputs))
print("Unique logged failures:", len(failed_ids))
print("Logged failed IDs:", failed_ids)

display(missing_outputs.head(20))


Expected requests: 2404
Downloaded raw CSVs: 2397
Missing raw CSVs: 7
Unique logged failures: 4
Logged failed IDs: ['fire_01258', 'fire_01546', 'fire_01554', 'fire_01560']


,id,lon,lat,start,end,downloaded
260,fire_00260,-124.196290,41.933230,2017-05-27,2017-06-25,False
1098,fire_01098,-122.361112,37.166674,2019-09-24,2019-10-23,False
1099,fire_01099,-121.685980,37.627820,2019-09-24,2019-10-23,False
1258,fire_01258,-124.199540,40.788040,2020-06-27,2020-07-26,False
1546,fire_01546,-120.277039,34.468742,2022-02-10,2022-03-11,False
1554,fire_01554,-117.744398,33.509464,2022-04-11,2022-05-10,False
1560,fire_01560,-123.664183,38.873375,2022-04-20,2022-05-19,False


## 3. Inspect one raw gridMET output

This confirms the raw PyGridMET files contain daily climate rows but no date column. That is expected for this extraction route.


In [4]:
raw_files = sorted(GRIDMET_OUTPUT_DIR.glob("*.csv"))

if not raw_files:
    raise FileNotFoundError(
        f"No raw gridMET CSVs found in {GRIDMET_OUTPUT_DIR}. "
        "Run PyGridMET extraction before aggregation."
    )

sample_path = raw_files[0]
sample = pd.read_csv(sample_path)

print("Sample file:", sample_path.name)
print("Shape:", sample.shape)
print("Columns:", sample.columns.tolist())

display(sample.head())
display(sample.tail())


Sample file: fire_00000.csv
Shape: (30, 13)
Columns: ['pr (mm)', 'tmmn (K)', 'tmmx (K)', 'rmin (%)', 'rmax (%)', 'vs (m/s)', 'srad (W/m2)', 'bi (-)', 'erc (-)', 'fm100 (%)', 'fm1000 (%)', 'vpd (kPa)', 'pet (mm)']


,pr (mm),tmmn (K),tmmx (K),rmin (%),rmax (%),vs (m/s),srad (W/m2),bi (-),erc (-),fm100 (%),fm1000 (%),vpd (kPa),pet (mm)
0,0.0,283.9,300.9,22.9,75.3,3.7,266.1,51.0,59.0,10.1,10.9,1.46,5.5
1,0.0,284.2,297.6,32.7,81.2,4.3,263.9,52.0,57.0,10.6,10.9,1.12,4.8
2,0.0,284.1,295.9,37.5,82.0,4.1,263.3,50.0,55.0,11.1,10.9,0.97,4.4
3,0.0,282.7,296.8,34.5,91.3,2.7,260.5,41.0,53.0,12.0,11.2,1.01,4.0
4,0.0,282.9,300.5,26.1,90.4,2.7,260.1,41.0,54.0,12.4,11.3,1.53,4.5


,pr (mm),tmmn (K),tmmx (K),rmin (%),rmax (%),vs (m/s),srad (W/m2),bi (-),erc (-),fm100 (%),fm1000 (%),vpd (kPa),pet (mm)
25,0.0,282.4,296.6,25.2,87.3,4.7,215.8,51.0,50.0,14.7,11.4,1.21,4.4
26,0.0,282.5,298.9,17.1,73.3,3.7,212.2,49.0,54.0,13.0,11.3,1.50,4.8
27,0.0,283.7,301.9,10.9,62.1,2.5,203.0,44.0,58.0,11.2,11.2,1.73,4.5
28,0.0,286.0,303.3,13.1,60.9,2.1,204.9,42.0,60.0,10.0,11.1,1.93,4.2
29,0.0,284.2,303.5,15.3,75.4,2.5,204.1,45.0,60.0,9.9,11.0,1.78,4.3


## 4. Aggregate raw gridMET files into 7-day, 14-day, and 30-day features

Because each raw file is the requested 30-day pre-fire window, this cell aggregates by row position:
- `df.tail(7)` for 7-day features
- `df.tail(14)` for 14-day features
- all available rows for 30-day features

This avoids the earlier bug where numeric weather columns were accidentally parsed as fake dates.


In [5]:
WINDOWS = [7, 14, 30]

def clean_col_name(col: str) -> str:
    """Convert gridMET column names with units into clean feature names."""
    col = str(col).strip()
    replacements = {
        " (%)": "_pct",
        " (mm)": "_mm",
        " (K)": "_K",
        " (m/s)": "_m_s",
        " (W/m2)": "_W_m2",
        " (kPa)": "_kPa",
        " (-)": "",
    }

    for old, new in replacements.items():
        col = col.replace(old, new)

    col = col.replace(" ", "_")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = re.sub(r"_+", "_", col)

    return col.strip("_")

def is_non_weather_column(col: str) -> bool:
    """Drop accidental index/date columns if they appear in raw files."""
    lowered = str(col).strip().lower()
    return (
        lowered.startswith("unnamed")
        or lowered in {"date", "time", "datetime", "day"}
        or lowered in {"x", "y", "lat", "latitude", "lon", "longitude"}
    )

def summarize_gridmet_file(file_path: Path) -> dict:
    """Summarize one raw gridMET CSV into windowed features."""
    gridmet_id = file_path.stem
    df = pd.read_csv(file_path)

    if df.empty:
        return {
            "gridmet_id": gridmet_id,
            "gridmet_rows_available": 0,
            "gridmet_empty_file": True,
        }

    df = df.rename(columns={col: clean_col_name(col) for col in df.columns})

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    numeric_cols = [
        col for col in df.columns
        if not is_non_weather_column(col) and df[col].notna().sum() > 0
    ]

    out = {
        "gridmet_id": gridmet_id,
        "gridmet_rows_available": len(df),
    }

    for window in WINDOWS:
        # If a file has fewer than 30 rows, this still uses all available rows up to that window.
        sub = df.tail(window).copy()
        out[f"n_days_{window}d"] = len(sub)

        for col in numeric_cols:
            values = pd.to_numeric(sub[col], errors="coerce")

            out[f"{col}_{window}d_mean"] = values.mean()
            out[f"{col}_{window}d_min"] = values.min()
            out[f"{col}_{window}d_max"] = values.max()
            out[f"{col}_{window}d_sum"] = values.sum()

    return out

rows = []
errors = []

for file_path in sorted(GRIDMET_OUTPUT_DIR.glob("*.csv")):
    try:
        rows.append(summarize_gridmet_file(file_path))
    except Exception as e:
        errors.append({
            "gridmet_id": file_path.stem,
            "error": str(e),
        })

gridmet_features = pd.DataFrame(rows)
aggregation_errors = pd.DataFrame(errors)

gridmet_features.to_csv(PROCESSED_DIR / "gridmet_features.csv", index=False)
aggregation_errors.to_csv(INTERIM_DIR / "gridmet_aggregation_errors.csv", index=False)

print("GridMET feature table:", gridmet_features.shape)
print("Aggregation errors:", aggregation_errors.shape)
print("Saved:", PROCESSED_DIR / "gridmet_features.csv")

display(gridmet_features.head())
display(aggregation_errors.head())


GridMET feature table: (2397, 161)
Aggregation errors: (0, 0)
Saved: C:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\gridmet_features.csv


,gridmet_id,gridmet_rows_available,n_days_7d,pr_mm_7d_mean,pr_mm_7d_min,pr_mm_7d_max,pr_mm_7d_sum,tmmn_K_7d_mean,tmmn_K_7d_min,tmmn_K_7d_max,...,fm1000_pct_30d_max,fm1000_pct_30d_sum,vpd_kPa_30d_mean,vpd_kPa_30d_min,vpd_kPa_30d_max,vpd_kPa_30d_sum,pet_mm_30d_mean,pet_mm_30d_min,pet_mm_30d_max,pet_mm_30d_sum
0,fire_00000,30,7,0.200000,0.0,1.4,1.4,283.557143,281.8,286.0,...,11.4,320.2,1.692667,0.29,3.26,50.78,4.783333,1.3,6.6,143.5
1,fire_00001,30,7,0.185714,0.0,1.3,1.3,284.571429,282.5,286.3,...,21.0,564.7,1.193667,0.49,2.27,35.81,4.606667,1.8,6.0,138.2
2,fire_00002,30,7,0.000000,0.0,0.0,0.0,285.457143,282.1,287.4,...,15.9,420.3,1.300333,0.58,2.36,39.01,5.533333,2.4,7.7,166.0
3,fire_00003,30,7,0.000000,0.0,0.0,0.0,281.714286,278.6,283.5,...,16.0,455.2,1.022333,0.40,2.02,30.67,4.973333,2.6,7.6,149.2
4,fire_00004,30,7,0.000000,0.0,0.0,0.0,283.828571,280.9,286.5,...,12.9,347.3,0.940667,0.36,1.91,28.22,4.633333,2.9,6.2,139.0


""


## 5. Merge gridMET features with the CAL FIRE lookup table

This creates the final modeling base:
- `data/processed/calfire_with_gridmet.csv`

It also adds:
- `fire_start_date`
- `year`
- `month`
- `log_acres`
- `size_tier`


In [6]:
features = pd.read_csv(PROCESSED_DIR / "gridmet_features.csv")
lookup = pd.read_csv(LOOKUP_PATH)

lookup["gridmet_id"] = lookup["gridmet_id"].astype(str)
features["gridmet_id"] = features["gridmet_id"].astype(str)

merged = lookup.merge(features, on="gridmet_id", how="inner")

if "StartedDateOnly" in merged.columns:
    merged["fire_start_date"] = pd.to_datetime(merged["StartedDateOnly"], errors="coerce")
elif "started_date" in merged.columns:
    merged["fire_start_date"] = pd.to_datetime(merged["started_date"], errors="coerce")
else:
    merged["fire_start_date"] = pd.NaT
    print("Warning: no StartedDateOnly or started_date column found.")

if "AcresBurned" not in merged.columns:
    raise ValueError("AcresBurned column not found.")

merged["AcresBurned"] = pd.to_numeric(merged["AcresBurned"], errors="coerce")

# Drop rows that cannot be labeled safely
pre_drop = len(merged)
merged = merged.dropna(subset=["AcresBurned", "fire_start_date"]).copy()
merged = merged[merged["AcresBurned"] > 0].copy()
print(f"Dropped {pre_drop - len(merged)} rows with missing/invalid acres or dates.")

merged["log_acres"] = np.log1p(merged["AcresBurned"])
merged["year"] = merged["fire_start_date"].dt.year
merged["month"] = merged["fire_start_date"].dt.month

# NWCG size classes:
# A: <= 0.25 acres
# B: > 0.25 and < 10 acres
# C: 10 and < 100 acres
# D: 100 and < 300 acres
# E: 300 and < 1,000 acres
# F: 1,000 and < 5,000 acres
# G: >= 5,000 acres
def nwcg_size_class(acres):
    if pd.isna(acres):
        return np.nan
    if acres <= 0.25:
        return "A"
    elif acres < 10:
        return "B"
    elif acres < 100:
        return "C"
    elif acres < 300:
        return "D"
    elif acres < 1000:
        return "E"
    elif acres < 5000:
        return "F"
    else:
        return "G"

def severity_tier(acres):
    if pd.isna(acres):
        return np.nan
    if acres < 100:
        return "Small"      # NWCG A-C
    elif acres < 1000:
        return "Medium"     # NWCG D-E
    elif acres < 5000:
        return "Large"      # NWCG F
    else:
        return "Extreme"    # NWCG G

merged["nwcg_size_class"] = merged["AcresBurned"].apply(nwcg_size_class)
merged["severity_tier"] = merged["AcresBurned"].apply(severity_tier)

# Optional backwards-compatible alias, but I would eventually remove this
merged["size_tier"] = merged["severity_tier"]

OUTPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet.csv"
merged.to_csv(OUTPUT_PATH, index=False)

print("Merged modeling table:", merged.shape)
print("Saved:", OUTPUT_PATH)

gridmet_cols = [c for c in merged.columns if any(w in c for w in ["7d", "14d", "30d"])]
print("GridMET feature columns:", len(gridmet_cols))

display(merged.head())

print("\nNWCG size class distribution:")
print(merged["nwcg_size_class"].value_counts().sort_index())

print("\nCollapsed severity tier distribution:")
print(merged["severity_tier"].value_counts())

C:\Users\chaud\AppData\Local\Temp\ipykernel_1872\2400816261.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["fire_start_date"] = pd.to_datetime(merged["StartedDateOnly"], errors="coerce")


Dropped 0 rows with missing/invalid acres or dates.
Merged modeling table: (2397, 174)
Saved: C:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet.csv
GridMET feature columns: 159


,gridmet_id,UniqueId,Name,StartedDateOnly,Latitude,Longitude,AcresBurned,gridmet_rows_available,n_days_7d,pr_mm_7d_mean,...,pet_mm_30d_min,pet_mm_30d_max,pet_mm_30d_sum,fire_start_date,log_acres,year,month,nwcg_size_class,severity_tier,size_tier
0,fire_00000,3135c367-1274-4cc4-9152-661f6fd7707e,Creek Fire,2016-10-10,38.409580,-122.431720,65.0,30,7,0.200000,...,1.3,6.6,143.5,2016-10-10,4.189655,2016,10,C,Small,Small
1,fire_00001,7106ba43-100b-41f9-acbf-9fb9e9ab9ef7,Taglio Fire,2016-04-24,37.217100,-121.080360,30.0,30,7,0.185714,...,1.8,6.0,138.2,2016-04-24,3.433987,2016,4,C,Small,Small
2,fire_00002,ffec404e-dea9-4175-8c50-8f2b0ca3efdd,Tulloch Fire,2016-05-30,37.927613,-120.528836,85.0,30,7,0.000000,...,2.4,7.7,166.0,2016-05-30,4.454347,2016,5,C,Small,Small
3,fire_00003,55fa95e6-f09e-42c4-889b-a4b5b63dccf2,Metz Fire,2016-05-22,36.381230,-121.200590,3876.0,30,7,0.000000,...,2.6,7.6,149.2,2016-05-22,8.262817,2016,5,F,Large,Large
4,fire_00004,ace1f464-7a70-4dcc-b8c5-d64fb1021d0e,Wheatland Fire,2016-05-23,34.276000,-118.354000,156.0,30,7,0.000000,...,2.9,6.2,139.0,2016-05-23,5.056246,2016,5,D,Medium,Medium



NWCG size class distribution:
nwcg_size_class
B      23
C    1288
D     448
E     272
F     200
G     166
Name: count, dtype: int64

Collapsed severity tier distribution:
severity_tier
Small      1311
Medium      720
Large       200
Extreme     166
Name: count, dtype: int64


## 6. Sanity checks

The key values to check:
- `n_days_7d` should usually equal 7
- `n_days_14d` should usually equal 14
- `n_days_30d` should usually equal 30
- important gridMET columns should not be mostly missing


In [7]:
check_cols = ["gridmet_id", "gridmet_rows_available", "n_days_7d", "n_days_14d", "n_days_30d"]
existing_check_cols = [c for c in check_cols if c in merged.columns]

print("Window-row counts:")
display(merged[existing_check_cols].describe(include="all"))

important_cols = [
    "tmmx_K_7d_mean",
    "tmmn_K_7d_mean",
    "vpd_kPa_7d_mean",
    "vs_m_s_7d_mean",
    "rmin_pct_7d_mean",
    "pr_mm_30d_sum",
    "erc_7d_mean",
    "bi_7d_mean",
    "fm100_pct_7d_mean",
    "fm1000_pct_30d_mean",
]

existing_important_cols = [c for c in important_cols if c in merged.columns]

if existing_important_cols:
    print("Missingness for important gridMET columns:")
    display(merged[existing_important_cols].isna().mean().sort_values(ascending=False))

preview_cols = ["Name", "fire_start_date", "AcresBurned", "size_tier"] + existing_important_cols
preview_cols = [c for c in preview_cols if c in merged.columns]

display(merged[preview_cols].head())


Window-row counts:


,gridmet_id,gridmet_rows_available,n_days_7d,n_days_14d,n_days_30d
count,2397,2397.000000,2397.0,2397.0,2397.000000
unique,2397,NaN,NaN,NaN,NaN
top,fire_00000,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN
mean,NaN,29.995828,7.0,14.0,29.995828
std,NaN,0.064469,0.0,0.0,0.064469
min,NaN,29.000000,7.0,14.0,29.000000
25%,NaN,30.000000,7.0,14.0,30.000000
50%,NaN,30.000000,7.0,14.0,30.000000
75%,NaN,30.000000,7.0,14.0,30.000000


Missingness for important gridMET columns:


tmmx_K_7d_mean         0.0
tmmn_K_7d_mean         0.0
vpd_kPa_7d_mean        0.0
vs_m_s_7d_mean         0.0
rmin_pct_7d_mean       0.0
pr_mm_30d_sum          0.0
erc_7d_mean            0.0
bi_7d_mean             0.0
fm100_pct_7d_mean      0.0
fm1000_pct_30d_mean    0.0
dtype: float64

,Name,fire_start_date,AcresBurned,size_tier,tmmx_K_7d_mean,tmmn_K_7d_mean,vpd_kPa_7d_mean,vs_m_s_7d_mean,rmin_pct_7d_mean,pr_mm_30d_sum,erc_7d_mean,bi_7d_mean,fm100_pct_7d_mean,fm1000_pct_30d_mean
0,Creek Fire,2016-10-10,65.0,Small,298.142857,283.557143,1.351429,3.071429,26.585714,2.3,51.857143,39.000000,12.814286,10.673333
1,Taglio Fire,2016-04-24,30.0,Small,302.100000,284.571429,1.652857,2.928571,25.842857,40.2,36.142857,25.000000,11.400000,18.823333
2,Tulloch Fire,2016-05-30,85.0,Small,300.557143,285.457143,1.415714,3.242857,30.228571,2.2,50.000000,40.857143,11.014286,14.010000
3,Metz Fire,2016-05-22,3876.0,Large,298.471429,281.714286,1.264286,5.057143,29.400000,9.2,42.428571,44.000000,12.800000,15.173333
4,Wheatland Fire,2016-05-23,156.0,Medium,295.985714,283.828571,0.955714,3.071429,36.528571,9.1,49.714286,39.714286,11.928571,11.576667


## 7. Files produced by this notebook

Final outputs:
- `data/processed/gridmet_features.csv`
- `data/processed/calfire_with_gridmet.csv`

Audit/log outputs:
- `data/interim/gridmet_missing_outputs.csv`
- `data/interim/gridmet_failed_ids_clean.csv`
- `data/interim/gridmet_aggregation_errors.csv`

Temporary batch folders are not needed after extraction and can be deleted.
